# ICD-O Diagnosis Code Mapping File

## Overview
This notebook creates a mapping table that pairs ICD-O diagnosis codes with their corresponding group. This mapping file is used to assist a script that preforms the following transformation, where the user supplies a list of diagnosis codes and receives the corresponding diagnosis group information.

---
<center>
  <h3>Input Example (2 columns) → Output Example (4 columns)</h1>
</center>

<table align="center">
<tr>
<td>

| code   | term              |
|--------|-------------------|
| 8000/0 | Neoplasm, benign  |
| 8000/0 | Tumor, benign     |
| 8000/1 | Tumor, NOS        |
| 8010/0 | Epithelial tumor  | 
</td>
<td align="center" style="font-size: 24px; padding: 20px;">

**→**

</td>
<td>

| code   | term              | prefix | group           |
|--------|-------------------|--------|-----------------|
| 8000/0 | Neoplasm, benign  | 800    | Neoplasms, NOS  |
| 8000/0 | Tumor, benign     | 800    | Neoplasms, NOS  |
| 8000/1 | Tumor, NOS        | 800    | Neoplasms, NOS  |
| 8010/0 | Epithelial tumor  | 801-804 | Epithelial neoplasms, NOS |
</td>
</tr>
</table>

---

## Final Deliverable
**File:** `icdo_mapping_file.tsv`

**Format:** Tab-separated values (TSV) with the following columns:
1. `code` - (integer) The original 4-digit/1-digit diagnosis code
2. `term` - (string) The diagnosis term
3. `prefix` - (integer) The 3-digit group code or range
4. `group` - (string) The diagnosis group name

### Step 0: Data Loading

**Source:** `ICD-O-3.2_MFin_17042019_web.xlsx`
- Contains ICD-O morphology codes and terms
- Includes hierarchy levels (Level 1, 2, Preferred, Synonym)
- Level 2 rows define groups (3-digits)
- Preferred/Synonym rows contain diagnosis codes (4-digits/1-digit)

In [140]:
import pandas as pd
from IPython.display import display, Markdown

In [141]:
file_path = '../data/code_terms.xlsx'
term_codes = pd.read_excel(file_path)

In [142]:
term_codes = term_codes.iloc[:,0:3]

In [143]:
term_codes.columns = ["code","level","term"]

In [144]:
term_codes.columns

Index(['code', 'level', 'term'], dtype='object')

In [145]:
term_codes[3:6]

,code,level,term
3,8000/0,Preferred,"Neoplasm, benign"
4,8000/0,Synonym,"Tumor, benign"
5,8000/0,Synonym,"Unclassified tumor, benign"


### Step 1: Extract Group Information
Filter the data to identify diagnosis groups and their ranges.

In [146]:
term_groups = term_codes[term_codes["level"]==2]
term_groups.columns = ["range","level","group"]
term_groups.head(3)

,range,level,group
2,800,2,"Neoplasms, NOS"
34,801-804,2,"Epithelial neoplasms, NOS"
88,805-808,2,Squamous cell neoplasms


In [147]:
# avoid setting with copy warning 
term_groups = term_groups.copy() 

# split the 'range' column
delta = term_groups['range'].str.split("-", expand = True)
term_groups['range_start'] = delta[0]
term_groups['range_end'] = delta[1]
term_groups['range_end'] = term_groups['range_end'].fillna(term_groups['range_start'])
term_groups.head(5)

,range,level,group,range_start,range_end
2,800,2,"Neoplasms, NOS",800,800
34,801-804,2,"Epithelial neoplasms, NOS",801,804
88,805-808,2,Squamous cell neoplasms,805,808
185,809-811,2,Basal cell neoplasms,809,811
238,812-813,2,Transitional cell papillomas and carcinomas,812,813


### Step 3: Create Lookup Glossary
Build a reference table where each 3-digit prefix maps to its group information.

In [148]:
# expand the ranges into individual rows
expanded_rows = []
for _ , row in term_groups.iterrows():
    start = int(row['range_start'])
    end = int(row['range_end'])
    for i in range(start, end + 1):
        new_row = row.copy()
        new_row['prefix'] = i
        expanded_rows.append(new_row)
glossary = pd.DataFrame(expanded_rows)

glossary.head()

,range,level,group,range_start,range_end,prefix
2,800,2,"Neoplasms, NOS",800,800,800
34,801-804,2,"Epithelial neoplasms, NOS",801,804,801
34,801-804,2,"Epithelial neoplasms, NOS",801,804,802
34,801-804,2,"Epithelial neoplasms, NOS",801,804,803
34,801-804,2,"Epithelial neoplasms, NOS",801,804,804


### Step 4: Map Diagnosis Codes to Groups
1. Extract first 3 digits from each diagnosis code
2. Look up the group information using the glossary
3. Merge diagnosis to group information in a new dataframe

In [149]:
# select individual diagnoses (Preferred/Synonym rows)
diagnosis_df = term_codes[term_codes['level'].isin(['Preferred', 'Synonym'])].copy()
# extract first 3 digits from diagnosis code
diagnosis_df['prefix'] = diagnosis_df['code'].str[:3].astype(int)
diagnosis_df

,code,level,term,prefix
3,8000/0,Preferred,"Neoplasm, benign",800
4,8000/0,Synonym,"Tumor, benign",800
5,8000/0,Synonym,"Unclassified tumor, benign",800
6,8000/1,Preferred,"Neoplasm, uncertain whether benign or malignant",800
7,8000/1,Synonym,"Neoplasm, NOS",800
...,...,...,...,...
2944,9987/3,Synonym,"Therapy-related myelodysplastic syndrome, epip...",998
2945,9989/3,Preferred,"Myelodysplastic syndrome, NOS",998
2946,9989/3,Synonym,Preleukemia,998
2947,9989/3,Synonym,Preleukemic syndrome,998


In [150]:
# merge with with group info from glossary 
mapping_file = diagnosis_df.merge(
    glossary[['prefix','range','group']],
    on = 'prefix',
    how = 'left'
    )
mapping_file

,code,level,term,prefix,range,group
0,8000/0,Preferred,"Neoplasm, benign",800,800,"Neoplasms, NOS"
1,8000/0,Synonym,"Tumor, benign",800,800,"Neoplasms, NOS"
2,8000/0,Synonym,"Unclassified tumor, benign",800,800,"Neoplasms, NOS"
3,8000/1,Preferred,"Neoplasm, uncertain whether benign or malignant",800,800,"Neoplasms, NOS"
4,8000/1,Synonym,"Neoplasm, NOS",800,800,"Neoplasms, NOS"
...,...,...,...,...,...,...
2326,9987/3,Synonym,"Therapy-related myelodysplastic syndrome, epip...",998,998-999,Myelodysplastic syndromes
2327,9989/3,Preferred,"Myelodysplastic syndrome, NOS",998,998-999,Myelodysplastic syndromes
2328,9989/3,Synonym,Preleukemia,998,998-999,Myelodysplastic syndromes
2329,9989/3,Synonym,Preleukemic syndrome,998,998-999,Myelodysplastic syndromes


### Step 5: Add Missing Codes
After testing the script, some codes were unmatched because they we not present in the initial raw data. We can add codes into the mapping file here.

In [167]:
file_path = '../data/missing_codes/Missing_Codes.xlsx'
missing_codes = pd.read_excel(file_path)
missing_codes

,code,prefix
0,8000/2,800
1,8004/1,800
2,8010/1,801
3,8011/2,801
4,8012/2,801
...,...,...
360,9836/3,983
361,9970/3,997
362,9971/3,997
363,9991/3,999


In [168]:
missing_code_info = mapping_file[mapping_file['level'] == 'Preferred']
missing_code_info = missing_code_info.drop(columns=['code'])

In [169]:
missing_codes = pd.merge(missing_codes, missing_code_info, on="prefix", how= "left")

In [170]:
col_order = mapping_file.columns
missing_codes = missing_codes[col_order]

In [171]:
missing_codes

,code,level,term,prefix,range,group
0,8000/2,Preferred,"Neoplasm, benign",800,800,"Neoplasms, NOS"
1,8000/2,Preferred,"Neoplasm, uncertain whether benign or malignant",800,800,"Neoplasms, NOS"
2,8000/2,Preferred,"Neoplasm, malignant",800,800,"Neoplasms, NOS"
3,8000/2,Preferred,"Neoplasm, metastatic",800,800,"Neoplasms, NOS"
4,8000/2,Preferred,"Neoplasm, malignant, uncertain whether primary...",800,800,"Neoplasms, NOS"
...,...,...,...,...,...,...
2989,9971/3,Preferred,"Lymphoproliferative disorder, NOS",997,997,Other hematological neoplasms
2990,9971/3,Preferred,"Post transplant lymphoproliferative disorder, NOS",997,997,Other hematological neoplasms
2991,9971/3,Preferred,"Myeloproliferative neoplasm, unclassifiable",997,997,Other hematological neoplasms
2992,9991/3,Preferred,Myelodysplastic syndrome with ring sideroblast...,999,998-999,Myelodysplastic syndromes


In [173]:
# since many terms will match a given prefix group, we will only keep the first match
missing_codes = missing_codes.drop_duplicates(subset=['code'], keep='first')
# remove the term because the term belongs to a different code in the same group
missing_codes['term'] = 'unknown'

In [174]:
missing_codes

,code,level,term,prefix,range,group
0,8000/2,Preferred,unknown,800,800,"Neoplasms, NOS"
13,8004/1,Preferred,unknown,800,800,"Neoplasms, NOS"
26,8010/1,Preferred,unknown,801,801-804,"Epithelial neoplasms, NOS"
37,8011/2,Preferred,unknown,801,801-804,"Epithelial neoplasms, NOS"
48,8012/2,Preferred,unknown,801,801-804,"Epithelial neoplasms, NOS"
...,...,...,...,...,...,...
2980,9836/3,Preferred,unknown,983,980-994,Leukemias
2986,9970/3,Preferred,unknown,997,997,Other hematological neoplasms
2989,9971/3,Preferred,unknown,997,997,Other hematological neoplasms
2992,9991/3,Preferred,unknown,999,998-999,Myelodysplastic syndromes


In [175]:
missing_codes[missing_codes['group'].isna()]

,code,level,term,prefix,range,group
2282,9150/0,NaN,unknown,915,NaN,NaN
2283,9150/1,NaN,unknown,915,NaN,NaN
2284,9150/3,NaN,unknown,915,NaN,NaN


In [177]:
# update terms for prefix 915
missing_codes.loc[missing_codes['prefix'] == 915, 'term'] = 'unknown'
missing_codes.loc[missing_codes['prefix'] == 915, 'level'] = 'Preferred'
missing_codes.loc[missing_codes['prefix'] == 915, 'range'] = 915
missing_codes.loc[missing_codes['prefix'] == 915, 'group'] = 'Vascular tumors'

# display the results immediately
missing_codes[missing_codes['prefix'] == 915]

,code,level,term,prefix,range,group
2282,9150/0,Preferred,unknown,915,915,Vascular tumors
2283,9150/1,Preferred,unknown,915,915,Vascular tumors
2284,9150/3,Preferred,unknown,915,915,Vascular tumors


In [178]:
# combine the master mapping file with newly fixed codes
updated_mapping = pd.concat([mapping_file, missing_codes], ignore_index=True)
updated_mapping

,code,level,term,prefix,range,group
0,8000/0,Preferred,"Neoplasm, benign",800,800,"Neoplasms, NOS"
1,8000/0,Synonym,"Tumor, benign",800,800,"Neoplasms, NOS"
2,8000/0,Synonym,"Unclassified tumor, benign",800,800,"Neoplasms, NOS"
3,8000/1,Preferred,"Neoplasm, uncertain whether benign or malignant",800,800,"Neoplasms, NOS"
4,8000/1,Synonym,"Neoplasm, NOS",800,800,"Neoplasms, NOS"
...,...,...,...,...,...,...
2691,9836/3,Preferred,unknown,983,980-994,Leukemias
2692,9970/3,Preferred,unknown,997,997,Other hematological neoplasms
2693,9971/3,Preferred,unknown,997,997,Other hematological neoplasms
2694,9991/3,Preferred,unknown,999,998-999,Myelodysplastic syndromes


### Step 6: Save Mapping File(s)
0. (icdo_)mapping_file is a general reference file containing level information in addition to group/diagnosis info
1. code_mapping_file is used for finding group information if the user supplies diagnosis codes
2. term_mapping_file is used for finding group information if the user supplies diagnosis terms

In [179]:
updated_mapping.to_csv('../data/icdo_mapping_file.tsv', sep='\t', index=False)
updated_mapping

,code,level,term,prefix,range,group
0,8000/0,Preferred,"Neoplasm, benign",800,800,"Neoplasms, NOS"
1,8000/0,Synonym,"Tumor, benign",800,800,"Neoplasms, NOS"
2,8000/0,Synonym,"Unclassified tumor, benign",800,800,"Neoplasms, NOS"
3,8000/1,Preferred,"Neoplasm, uncertain whether benign or malignant",800,800,"Neoplasms, NOS"
4,8000/1,Synonym,"Neoplasm, NOS",800,800,"Neoplasms, NOS"
...,...,...,...,...,...,...
2691,9836/3,Preferred,unknown,983,980-994,Leukemias
2692,9970/3,Preferred,unknown,997,997,Other hematological neoplasms
2693,9971/3,Preferred,unknown,997,997,Other hematological neoplasms
2694,9991/3,Preferred,unknown,999,998-999,Myelodysplastic syndromes


In [180]:
# filter out synonyms
code_mapping_file = updated_mapping[updated_mapping['level'] == 'Preferred']
code_mapping_file = code_mapping_file.drop(columns=['level','prefix'])

# save mapping file for codes
code_mapping_file.to_csv('../data/code_mapping_file.tsv', sep='\t', index=False)
code_mapping_file

,code,term,range,group
0,8000/0,"Neoplasm, benign",800,"Neoplasms, NOS"
3,8000/1,"Neoplasm, uncertain whether benign or malignant",800,"Neoplasms, NOS"
8,8000/3,"Neoplasm, malignant",800,"Neoplasms, NOS"
14,8000/6,"Neoplasm, metastatic",800,"Neoplasms, NOS"
19,8000/9,"Neoplasm, malignant, uncertain whether primary...",800,"Neoplasms, NOS"
...,...,...,...,...
2691,9836/3,unknown,980-994,Leukemias
2692,9970/3,unknown,997,Other hematological neoplasms
2693,9971/3,unknown,997,Other hematological neoplasms
2694,9991/3,unknown,998-999,Myelodysplastic syndromes


In [182]:
# save mapping file for terms
term_mapping_file = mapping_file.drop(columns=['level','prefix'])
term_mapping_file.to_csv('../data/term_mapping_file.tsv', sep='\t', index=False)
term_mapping_file

,code,term,range,group
0,8000/0,"Neoplasm, benign",800,"Neoplasms, NOS"
1,8000/0,"Tumor, benign",800,"Neoplasms, NOS"
2,8000/0,"Unclassified tumor, benign",800,"Neoplasms, NOS"
3,8000/1,"Neoplasm, uncertain whether benign or malignant",800,"Neoplasms, NOS"
4,8000/1,"Neoplasm, NOS",800,"Neoplasms, NOS"
...,...,...,...,...
2326,9987/3,"Therapy-related myelodysplastic syndrome, epip...",998-999,Myelodysplastic syndromes
2327,9989/3,"Myelodysplastic syndrome, NOS",998-999,Myelodysplastic syndromes
2328,9989/3,Preleukemia,998-999,Myelodysplastic syndromes
2329,9989/3,Preleukemic syndrome,998-999,Myelodysplastic syndromes
